### Setup required directories

In [ ]:
!mkdir -p src
!mkdir -p /kaggle/working/models
!mkdir -p /kaggle/working/eyes_data
!pip uninstall -y tensorflow
!pip install mediapipe==0.9.3 tqdm -q

In [ ]:
%%writefile src/config.py
IMG_SIZE   = 224   # EfficientNet-B3 works best at 224+, not 112
BATCH_SIZE = 32    # safe for GPU; increase to 64 only if no OOM
EPOCHS     = 50

LR_BACKBONE   = 5e-6  # very small — EfficientNet is pretrained
LR_HEAD       = 5e-4
WEIGHT_DECAY  = 1e-4
FREEZE_EPOCHS = 5     # freeze backbone longer before fine-tuning
SUBSAMPLE     = 1

TEST_PARTICIPANT = "participant_5"

SCREEN_W = 1920
SCREEN_H = 1080

DATA_DIR   = "/kaggle/working/eyes_data"
MODELS_DIR = "/kaggle/working/models"

In [ ]:
%%writefile src/utils.py
import os
import pandas as pd
from src.config import SUBSAMPLE

def build_dataframe(df, frame_dir, participant):
    df = df.copy()
    df["VID_FRAME"] = df["VID_FRAME"].astype(int)
    df["image_path"] = df["VID_FRAME"].apply(
        lambda fid: os.path.join(frame_dir, f"{fid}.jpg")
    )
    df["participant"] = participant
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
    if SUBSAMPLE > 1:
        df = df.iloc[::SUBSAMPLE].reset_index(drop=True)
    return df[["image_path", "VID_FRAME", "FPOGX", "FPOGY", "participant"]]

In [ ]:
%%writefile src/preprocess.py
import os
import pandas as pd

def extract_frames(video_path, csv_path, frame_dir):
    df = pd.read_csv(csv_path)
    time_col = next((c for c in df.columns if "TIME" in c.upper()), None)
    
    # --- LATEST CHANGE ---
    # Safely drop blanks and zero-out the start time for perfect video sync
    df = df[df["FPOGV"] == 1].copy()
    df = df.dropna(subset=[time_col, "FPOGX", "FPOGY"])
    df[time_col] = df[time_col] - df[time_col].min()

    df = df[
        (df["FPOGX"] >= 0) & (df["FPOGX"] <= 1) &
        (df["FPOGY"] >= 0) & (df["FPOGY"] <= 1)
    ].reset_index(drop=True)

    fps = 30.0 # Default to 30 instead of 10
    try:
        import cv2
        cap = cv2.VideoCapture(video_path)
        detected = cap.get(cv2.CAP_PROP_FPS)
        if detected > 0:
            fps = detected
        cap.release()
    except Exception:
        pass

    df["VID_FRAME"] = (df[time_col] * fps).round().astype(int)
    df = df.groupby("VID_FRAME", as_index=False)[["FPOGX", "FPOGY"]].mean()
    return df

In [ ]:
%%writefile src/dataset.py
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from src.config import IMG_SIZE

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class GazeDataset(Dataset):
    def __init__(self, df, landmarks_dict, augment=False):
        self.df = df.reset_index(drop=True)
        self.landmarks_dict = landmarks_dict

        self.lm_size = 0
        for v in self.landmarks_dict.values():
            if v and len(v) > 0:
                self.lm_size = len(v)
                break

        aug_list = []
        if augment:
            aug_list = [
                # Color-only augmentation — safe for gaze estimation
                # Spatial transforms (RandomAffine, RandomFlip) are removed
                # because they misalign the face image with the landmarks
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
            ]

        self.face_transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            *aug_list,
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        image    = Image.open(row["image_path"]).convert("RGB")
        face_img = self.face_transform(image)

        lms = self.landmarks_dict.get(row["image_path"])
        lms_tensor = (
            torch.tensor(lms, dtype=torch.float32)
            if lms is not None
            else torch.zeros(self.lm_size, dtype=torch.float32)
        )

        target = torch.tensor([float(row["FPOGX"]), float(row["FPOGY"])],
                               dtype=torch.float32)
        return face_img, lms_tensor, target

In [ ]:
%%writefile src/model.py
import torch
import torch.nn as nn
from torchvision import models

class DualBranchGazeModel(nn.Module):
    def __init__(self, landmark_dim):
        super().__init__()
        try:
            weights = models.EfficientNet_B3_Weights.DEFAULT
            face_net = models.efficientnet_b3(weights=weights)
        except AttributeError:
            face_net = models.efficientnet_b3(pretrained=True)
            
        face_in = face_net.classifier[1].in_features  
        face_net.classifier = nn.Identity()
        self.face_branch = face_net

        self.lm_branch = nn.Sequential(
            nn.Linear(landmark_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True)
        )

        self.fusion_head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(face_in + 128, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(512, 2),
            nn.Sigmoid(),
        )

    def forward(self, face_img, lms):
        return self.fusion_head(
            torch.cat((self.face_branch(face_img), self.lm_branch(lms)), dim=1)
        )

def get_model(landmark_dim):
    return DualBranchGazeModel(landmark_dim)

In [ ]:
%%writefile src/train.py
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.dataset import GazeDataset
from src.config import BATCH_SIZE, EPOCHS, LR_BACKBONE, LR_HEAD, WEIGHT_DECAY, FREEZE_EPOCHS

# --- TPU SPECIFIC IMPORTS ---
import torch_xla.core.xla_model as xm

# --- GPU SPECIFIC IMPORTS (Commented for Local GPU Use) ---
# from torch.amp import autocast, GradScaler

def set_backbone_requires_grad(model, requires_grad):
    for param in model.face_branch.parameters():
        param.requires_grad = requires_grad

def train_model(model, train_df, landmarks_dict, save_path=None):
    train_dfs, val_dfs = [], []
    for p, group in train_df.groupby("participant"):
        group = group.sort_values("VID_FRAME").reset_index(drop=True)
        split = int(len(group) * 0.9)
        train_dfs.append(group.iloc[:split])
        val_dfs.append(group.iloc[split:])

    train_sub_df = pd.concat(train_dfs, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    val_sub_df   = pd.concat(val_dfs, ignore_index=True)

    train_dataset = GazeDataset(train_sub_df, landmarks_dict, augment=True)
    val_dataset   = GazeDataset(val_sub_df,   landmarks_dict, augment=False)

    # ==========================================
    # DEVICE TOGGLE: TPU vs GPU
    # ==========================================
    
    # --- ACTIVE: TPU Setup ---
    # This command searches Kaggle for the TPU core and binds the model to it.
    device = xm.xla_device() 
    pin = False 
    print(f"  Device: {device} (TPU Active)")
    
    # --- COMMENTED: Local GPU Setup ---
    # device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # use_amp = device.type == "cuda"
    # pin     = device.type == "cuda"
    # scaler  = GradScaler("cuda", enabled=use_amp)
    # print(f"  Device: {device} | AMP: {use_amp}")
    # ==========================================

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=pin)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)

    model = model.to(device)
    set_backbone_requires_grad(model, False)

    criterion = nn.SmoothL1Loss()
    optimizer = torch.optim.AdamW([
        {"params": model.face_branch.parameters(),  "lr": LR_BACKBONE},
        {"params": model.lm_branch.parameters(),    "lr": LR_HEAD},
        {"params": model.fusion_head.parameters(),  "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1)

    best_val_loss = float("inf")
    best_state    = None

    # Gradient accumulation handles small VRAM. Since TPU has 128GB, we leave it at 1.
    # If moving back to a 4GB GPU, change BATCH_SIZE to 16 in config and set this to 4.
    accumulation_steps = 1 

    for epoch in range(EPOCHS):
        if epoch == FREEZE_EPOCHS:
            set_backbone_requires_grad(model, True)

        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        
        for i, (face_img, lms, targets) in enumerate(train_loader):
            face_img = face_img.to(device)
            lms      = lms.to(device)
            targets  = targets.to(device)
            
            # --- ACTIVE: TPU Forward/Backward Pass ---
            outputs = model(face_img, lms)
            loss = criterion(outputs, targets)
            loss = loss / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0:
                # xm.optimizer_step() triggers the XLA graph execution barrier
                xm.optimizer_step(optimizer)
                optimizer.zero_grad()
            
            # --- COMMENTED: Local GPU Forward/Backward Pass ---
            # with autocast("cuda", enabled=use_amp):
            #     loss = criterion(model(face_img, lms), targets)
            # loss = loss / accumulation_steps
            # scaler.scale(loss).backward()
            # if (i + 1) % accumulation_steps == 0:
            #     scaler.step(optimizer)
            #     scaler.update()
            #     optimizer.zero_grad()

            train_loss += loss.item() * accumulation_steps

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for face_img, lms, targets in val_loader:
                face_img = face_img.to(device)
                lms      = lms.to(device)
                targets  = targets.to(device)
                
                # --- ACTIVE: TPU Eval ---
                val_loss += criterion(model(face_img, lms), targets).item()
                
                # --- COMMENTED: GPU Eval ---
                # with autocast("cuda", enabled=use_amp):
                #     val_loss += criterion(model(face_img, lms), targets).item()

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        scheduler.step()

        print(f"  Epoch {epoch+1:02d}/{EPOCHS}  Train: {avg_train:.6f}  Val: {avg_val:.6f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            # Move state dict to CPU before saving to avoid memory leaks on XLA devices
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if save_path:
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                torch.save(best_state, save_path)

    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return model

In [ ]:
%%writefile src/evaluate.py
import torch
import numpy as np
from torch.utils.data import DataLoader
from src.config import SCREEN_W, SCREEN_H, BATCH_SIZE
import torch_xla.core.xla_model as xm

def evaluate(model, test_dataset):
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # --- LATEST CHANGE: TPU Evaluation Device ---
    device = xm.xla_device()
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # (For GPU)
    
    model  = model.to(device)
    model.eval()

    x_errors, y_errors = [], []
    with torch.no_grad():
        for face_img, lms, targets in test_loader:
            face_img = face_img.to(device)
            lms      = lms.to(device)
            preds    = model(face_img, lms).cpu().numpy()
            targets  = targets.numpy()
            for pred, target in zip(preds, targets):
                x_errors.append(abs(pred[0] - target[0]))
                y_errors.append(abs(pred[1] - target[1]))

    x_errors = np.array(x_errors)
    y_errors = np.array(y_errors)
    mae_x, mae_y = float(np.mean(x_errors)), float(np.mean(y_errors))
    px_x,  px_y  = mae_x * SCREEN_W, mae_y * SCREEN_H
    pixel_error  = float(np.mean(np.sqrt((x_errors * SCREEN_W)**2 + (y_errors * SCREEN_H)**2)))
    norm_error   = float(np.mean(np.sqrt(x_errors**2 + y_errors**2)))

    print(f"\n  {'Metric':<30} {'X (horizontal)':>18} {'Y (vertical)':>16}")
    print(f"  {'-'*66}")
    print(f"  {'Mean Absolute Error (norm)':<30} {mae_x:>17.4f}  {mae_y:>15.4f}")
    print(f"  {'Error as % of screen':<30} {mae_x*100:>16.1f}%  {mae_y*100:>14.1f}%")
    print(f"  {'Pixel error (1920x1080)':<30} {px_x:>15.1f}px  {px_y:>13.1f}px")
    print(f"  {'-'*66}")
    print(f"  Overall Euclidean pixel error : {pixel_error:.1f} px")
    print(f"  Overall normalized error      : {norm_error:.4f}")
    return norm_error, pixel_error

In [ ]:
%%writefile prepare_data.py
import os
import cv2
import json
import shutil
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

INPUT_DIR  = "/kaggle/input/datasets/yadavudbhav/data-gaze/data"
OUTPUT_DIR = "/kaggle/working/eyes_data"
mp_face = mp.solutions.face_mesh

def process_image_for_landmarks(img, face_mesh):
    if img is None: return None, None
    
    # --- LATEST CHANGE: Perfect Crop Isolation ---
    camera_part = img[0:120, 0:160]
    face_only = camera_part[0:50, 0:160] 
    
    results = face_mesh.process(cv2.cvtColor(face_only, cv2.COLOR_BGR2RGB))
    
    lms_list = None
    if results.multi_face_landmarks:
        lms = results.multi_face_landmarks[0].landmark
        lms_list = [val for lm in lms for val in [lm.x, lm.y, lm.z]]
        
    return camera_part, lms_list

def prepare_participant(p, face_mesh):
    in_p  = os.path.join(INPUT_DIR,  p)
    out_p = os.path.join(OUTPUT_DIR, p)
    out_f = os.path.join(out_p, "frames")
    os.makedirs(out_f, exist_ok=True)

    csv_path, video_path = None, None
    for fname in os.listdir(in_p):
        if fname.endswith(".csv"):
            csv_path = os.path.join(in_p, fname)
            shutil.copy(csv_path, os.path.join(out_p, fname))
        elif fname.endswith(".avi"):
            video_path = os.path.join(in_p, fname)
            open(os.path.join(out_p, fname), "w").close()

    in_f = os.path.join(in_p, "frames")
    p_lms = {}
    
    # Path A: Images already exist
    if os.path.exists(in_f):
        jpegs = [f for f in os.listdir(in_f) if f.endswith(".jpg")]
        for fname in tqdm(jpegs, desc=f"Processing {p} (images)"):
            src = os.path.join(in_f, fname)
            dst = os.path.join(out_f, fname)
            
            img = cv2.imread(src)
            camera_part, lms = process_image_for_landmarks(img, face_mesh)
            
            if camera_part is not None:
                cv2.imwrite(dst, camera_part)
                if lms: p_lms[dst] = lms
        print(f"  {p}: {len(jpegs)} frames processed")

    # Path B: No images, must extract from video
    elif video_path and csv_path:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        df = pd.read_csv(csv_path)
        
        # --- LATEST CHANGE: Sync Data ---
        time_col = next((c for c in df.columns if "TIME" in c.upper()), None)
        df_valid = df[df["FPOGV"] == 1].copy()
        df_valid = df_valid.dropna(subset=[time_col, "FPOGX", "FPOGY"])
        df_valid[time_col] = df_valid[time_col] - df_valid[time_col].min()
        valid_frames = set((df_valid[time_col] * fps).round().astype(int).tolist())
        
        frame_id, saved, total_vframes = 0, 0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        with tqdm(total=total_vframes, desc=f"Processing {p} (video)") as pbar:
            while True:
                success, frame = cap.read()
                if not success: break
                
                if frame_id in valid_frames:
                    dst = os.path.join(out_f, f"{frame_id}.jpg")
                    camera_part, lms = process_image_for_landmarks(frame, face_mesh)
                    
                    if camera_part is not None:
                        cv2.imwrite(dst, camera_part)
                        if lms: p_lms[dst] = lms
                        saved += 1
                frame_id += 1
                pbar.update(1)
        cap.release()
        print(f"  {p}: {saved} frames extracted and processed")
    else:
        print(f"  WARNING: no frames or video found for {p}")

    with open(os.path.join(out_p, "lms.json"), "w") as f:
        json.dump(p_lms, f)

if __name__ == "__main__":
    participants = sorted([p for p in os.listdir(INPUT_DIR) if p.startswith("participant")])
    print(f"Participants: {participants}")
    with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.2) as face_mesh:
        for p in participants:
            prepare_participant(p, face_mesh)
    print("Extraction completed.")

In [ ]:
%%writefile main.py
import os
import json
import pandas as pd
from src.preprocess import extract_frames
from src.utils import build_dataframe
from src.dataset import GazeDataset
from src.model import get_model
from src.train import train_model
from src.evaluate import evaluate
from src.config import DATA_DIR, MODELS_DIR, TEST_PARTICIPANT

def find_files(p_path):
    video_path, csv_path = None, None
    for fname in os.listdir(p_path):
        if fname.endswith(".avi"): video_path = os.path.join(p_path, fname)
        elif fname.endswith(".csv"): csv_path  = os.path.join(p_path, fname)
    return video_path, csv_path

def main():
    os.makedirs(MODELS_DIR, exist_ok=True)
    participants = sorted([p for p in os.listdir(DATA_DIR) if p.startswith("participant")])
    print(f"Participants : {participants}")
    print(f"Test subject : {TEST_PARTICIPANT}\n")

    all_data = {}
    global_landmarks = {}

    for p in participants:
        p_path    = os.path.join(DATA_DIR, p)
        frame_dir = os.path.join(p_path, "frames")
        video_path, csv_path = find_files(p_path)
        
        json_path = os.path.join(p_path, "lms.json")
        if os.path.exists(json_path):
            with open(json_path, 'r') as f: global_landmarks.update(json.load(f))

        df = extract_frames(video_path, csv_path, frame_dir)
        df = build_dataframe(df, frame_dir, p)
        print(f"  [{p}] {len(df)} samples")
        all_data[p] = df

    train_participants = [p for p in participants if p != TEST_PARTICIPANT]
    train_df = pd.concat([all_data[p] for p in train_participants], ignore_index=True)
    test_df  = all_data[TEST_PARTICIPANT]
    print(f"\nTrain: {len(train_df)}  |  Test: {len(test_df)}\n")

    sample_lms = next(iter(global_landmarks.values()))
    model     = get_model(len(sample_lms))
    save_path = os.path.join(MODELS_DIR, "final_model_dual.pth")
    model     = train_model(model, train_df, global_landmarks, save_path=save_path)

    print("\n" + "="*60)
    print(f"Evaluation on held-out [{TEST_PARTICIPANT}]")
    print("="*60)
    test_dataset = GazeDataset(test_df, global_landmarks, augment=False)
    evaluate(model, test_dataset)
    print(f"\nModel saved -> {save_path}")

if __name__ == "__main__":
    main()

In [ ]:
import glob

all_py_files = glob.glob("*.py") + glob.glob("src/*.py")
for filepath in all_py_files:
    with open(filepath, 'r', encoding='utf-8') as f: content = f.read()
    with open(filepath, 'w', encoding='utf-8') as f: f.write(content.replace('\xa0', ' '))
        
print("Ghost spaces scrubbed successfully!")

In [ ]:
!python -u prepare_data.py
!python -u main.py